# Research 19 — LightGBM balanced class weights

**Hypothesis:** the macro-F1 objective may benefit from inverse-frequency class weights. CatBoost's balanced variant was weak, but LightGBM can behave differently. Run one fixed 3-fold OOF experiment with `class_weight='balanced'`; compare standalone, fixed 80/20, and fixed four-way blends. No tuning sweep; Test is not loaded.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    values = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(values).all() and np.all(values >= 0)
    sums = values.sum(axis=1, keepdims=True)
    assert np.all(sums > 0)
    return values / sums


p_base = load_probs("segment-oof.csv")
p_xgb = load_probs("segment-xgboost-oof.csv")
p_balanced_cat = load_probs("segment-balanced-catboost-oof.csv")


def features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (numerator, denominator) in ratios.items():
        x[name] = x[numerator] / (x[denominator].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = features(train)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
p_candidate = np.zeros((len(y), 4), dtype=float)
started = perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    x_fit, x_valid = X.iloc[fit_idx].copy(), X.iloc[valid_idx].copy()
    for col in cat_cols:
        dtype = pd.CategoricalDtype(categories=sorted(x_fit[col].dropna().unique().tolist()))
        x_fit[col] = x_fit[col].astype(dtype)
        x_valid[col] = x_valid[col].astype(dtype)
    model = LGBMClassifier(
        objective="multiclass",
        class_weight="balanced",
        n_estimators=440,
        learning_rate=0.063863707925569,
        num_leaves=59,
        max_depth=4,
        min_child_samples=81,
        reg_lambda=0.12942956115511214,
        colsample_bytree=0.7975990992289793,
        random_state=42,
        n_jobs=2,
        verbosity=-1,
    )
    model.fit(x_fit, y[fit_idx], categorical_feature=cat_cols)
    p_candidate[valid_idx] = model.predict_proba(x_valid)
    print(
        f"Fold {fold}: macro F1={f1_score(y[valid_idx], p_candidate[valid_idx].argmax(1), average='macro'):.6f}",
        flush=True,
    )

p_blend = 0.8 * p_base + 0.2 * p_candidate
p_fourway = 0.7 * p_base + 0.1 * p_xgb + 0.1 * p_balanced_cat + 0.1 * p_candidate
candidates = {
    "Current reference": p_base,
    "Balanced-weight LightGBM": p_candidate,
    "80/20 current + balanced-weight LGBM": p_blend,
    "70/10/10/10 with balanced-weight LGBM": p_fourway,
}
folds = list(cv.split(X, y))
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
base_folds = results.loc[
    results.candidate == "Current reference", [f"fold_{fold}_macro_f1" for fold in range(1, 4)]
].iloc[0]
for fold in range(1, 4):
    results[f"fold_{fold}_delta"] = (
        results[f"fold_{fold}_macro_f1"] - base_folds[f"fold_{fold}_macro_f1"]
    )
display(results.round(6))
results.to_csv(ROOT / "assets/segment-lightgbm-balanced-weights.csv", index=False)
for suffix, probs in [
    ("balanced-weights", p_candidate),
    ("balanced-weights-blend", p_blend),
    ("balanced-weights-fourway", p_fourway),
]:
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / f"predictions/segment-lightgbm-{suffix}-oof.csv", index=False
    )
print(f"Elapsed seconds={perf_counter() - started:.1f}. Training OOF only; Test not loaded.")

Fold 1: macro F1=0.742874


Fold 2: macro F1=0.741536


Fold 3: macro F1=0.746208


,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1,fold_1_delta,fold_2_delta,fold_3_delta
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.000000,0.000000,0.000000
1,Balanced-weight LightGBM,0.743547,0.755451,0.742874,0.741536,0.746208,-0.006572,-0.008285,-0.007182
2,80/20 current + balanced-weight LGBM,0.751550,0.764986,0.749726,0.750842,0.754051,0.000280,0.001021,0.000661
3,70/10/10/10 with balanced-weight LGBM,0.751479,0.764955,0.750146,0.750910,0.753350,0.000701,0.001089,-0.000040


Elapsed seconds=72.9. Training OOF only; Test not loaded.


## Decision

Treat class weighting as one predeclared macro-F1 hypothesis. Compare fold deltas and both metrics; reject it if gains are not consistent. No Test or submission outputs are produced.